# 实验结果深度分析

对推荐系统实验结果进行深入分析：模型对比可视化、消融分析、统计显著性检验、错误案例研究、覆盖度/多样性权衡，以及最终结论。

In [ ]:
from __future__ import annotations
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

from src.config import REPORTS_DIR

%matplotlib inline

## 1. 加载预计算结果

In [ ]:
def load_report(name: str) -> pd.DataFrame | None:
    path = REPORTS_DIR / name
    if path.exists():
        return pd.read_csv(path)
    print(f"  [跳过] {name} 不存在，请先运行 02_baseline_experiments.ipynb")
    return None

evaluation = load_report("model_comparison.csv")
ablation = load_report("ablation_results.csv")
segments = load_report("user_segment_results.csv")
bias = load_report("popularity_bias.csv")

available = sum(1 for d in [evaluation, ablation, segments, bias] if d is not None)
print(f"成功加载 {available}/4 个报告文件")

## 2. 模型综合对比（雷达图）

In [ ]:
if evaluation is not None:
    radar_metrics = ["precision_at_k", "recall_at_k", "ndcg_at_k", "hit_rate_at_k", "map_at_k", "mrr_at_k", "coverage", "diversity"]
    radar_metrics = [m for m in radar_metrics if m in evaluation.columns]

    categories = radar_metrics
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

    colors = plt.cm.tab10(np.linspace(0, 1, len(evaluation)))
    for (_, row), color in zip(evaluation.iterrows(), colors):
        values = [row[m] for m in radar_metrics]
        values += values[:1]
        ax.fill(angles, values, alpha=0.1, color=color)
        ax.plot(angles, values, "o-", linewidth=2, label=row["model"], color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_title("Model Comparison: Radar Chart", fontsize=14, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()

## 3. 消融分析深入

In [ ]:
if ablation is not None:
    print("=== 消融实验排序 (NDCG@K) ===")
    display(ablation.sort_values("ndcg_at_k", ascending=False))

    ablation_metrics = ["precision_at_k", "recall_at_k", "ndcg_at_k", "hit_rate_at_k", "map_at_k"]
    ablation_metrics = [m for m in ablation_metrics if m in ablation.columns]

    fig, axes = plt.subplots(1, len(ablation_metrics), figsize=(4 * len(ablation_metrics), 5))
    for i, metric in enumerate(ablation_metrics):
        ax = axes[i]
        sorted_df = ablation.sort_values(metric, ascending=True)
        ax.barh(sorted_df["ablation_setting"], sorted_df[metric], color="steelblue")
        ax.set_title(metric)
        ax.set_xlabel("score")
    plt.suptitle("Ablation: Component Contribution", fontsize=14)
    plt.tight_layout()
    plt.show()

    best = ablation.sort_values("ndcg_at_k", ascending=False).iloc[0]
    pop_only = ablation[ablation["ablation_setting"] == "popularity_only"]
    cf_only = ablation[ablation["ablation_setting"] == "cf_only_item"]
    if not pop_only.empty and not cf_only.empty:
        ndcg_gain = (best["ndcg_at_k"] - pop_only["ndcg_at_k"].values[0]) / pop_only["ndcg_at_k"].values[0] * 100
        print(f"\n混合模型较纯流行度的 NDCG 提升: {ndcg_gain:.1f}%")

## 4. 用户分群性能分析

In [ ]:
if segments is not None:
    for segment_name, group in segments.groupby("activity_segment"):
        print(f"\n=== {segment_name} ===")
        best = group.sort_values("ndcg_at_k", ascending=False).iloc[0]
        worst = group.sort_values("ndcg_at_k", ascending=True).iloc[0]
        print(f"  Best: {best['model']} (NDCG={best['ndcg_at_k']:.4f})")
        print(f"  Worst: {worst['model']} (NDCG={worst['ndcg_at_k']:.4f})")
        print(f"  Gap: {best['ndcg_at_k'] - worst['ndcg_at_k']:.4f}")

    pivot = segments.pivot_table(index="model", columns="activity_segment", values="ndcg_at_k")
    print("\nNDCG@K 透视表:")
    display(pivot)

## 5. 统计显著性检验（Iteration 2 完成后启用）

In [ ]:
HAS_SIGNIFICANCE = False
try:
    from src.evaluation.significance import model_significance_report
    HAS_SIGNIFICANCE = True
except ImportError:
    pass

if HAS_SIGNIFICANCE:
    from src.data.feature_engineering import build_feature_store
    from src.data.preprocess import clean_movies, clean_ratings, clean_tags, leave_one_out_split
    from src.data.load_data import make_sample_movielens
    from src.models import build_default_models

    data = make_sample_movielens()
    movies = clean_movies(data["movies"])
    ratings = clean_ratings(data["ratings"])
    tags = clean_tags(data["tags"])
    movies = movies[movies["movieId"].isin(ratings["movieId"])]
    train, test = leave_one_out_split(ratings)
    features = build_feature_store(train, movies, tags)

    models = build_default_models()
    for m in models.values():
        m.fit(features)

    sig_report = model_significance_report(models, features, test, k=5)
    display(sig_report)
else:
    print("significance 模块尚未实现（Iteration 2）。请运行 Iteration 2 后再执行此 cell。")

## 6. 错误案例分析

In [ ]:
if evaluation is not None:
    best_model_name = evaluation.sort_values("ndcg_at_k", ascending=False).iloc[0]["model"]
    print(f"最佳模型: {best_model_name}")

    from src.data.load_data import make_sample_movielens
    from src.data.feature_engineering import build_feature_store
    from src.data.preprocess import clean_movies, clean_ratings, clean_tags, leave_one_out_split
    from src.models import build_default_models

    data = make_sample_movielens()
    movies = clean_movies(data["movies"])
    ratings = clean_ratings(data["ratings"])
    tags = clean_tags(data["tags"])
    movies = movies[movies["movieId"].isin(ratings["movieId"])]
    train, test = leave_one_out_split(ratings)
    features = build_feature_store(train, movies, tags)

    models = build_default_models()
    for m in models.values():
        m.fit(features)

    from src.evaluation.metrics import relevant_by_user
    relevant = relevant_by_user(test)

    print("\n分析每位用户的推荐质量:")
    best_model = models[best_model_name]
    for user_id in sorted(relevant)[:5]:
        recs = best_model.recommend(user_id, k=5, exclude_seen=True)
        truth = relevant[user_id]
        hits = set(recs["movieId"].astype(int)) & truth
        status = "HIT" if hits else "MISS"
        print(f"\n  User {user_id} [{status}]:")
        print(f"    Relevant: {truth}")
        print(f"    Recommended: {recs['movieId'].astype(int).tolist()}")
        if not hits:
            user_rated = set(features.ratings[features.ratings["userId"] == user_id]["movieId"])
            print(f"    User has rated {len(user_rated)} movies in training")

## 7. 覆盖度 / 多样性权衡

In [ ]:
if evaluation is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    for _, row in evaluation.iterrows():
        ax.scatter(row["coverage"], row["diversity"], s=200, alpha=0.7, label=row["model"])
        ax.annotate(row["model"], (row["coverage"], row["diversity"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=10)
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Diversity")
    ax.set_title("Coverage vs Diversity Trade-off")
    ax.legend()
    plt.tight_layout()
    plt.show()

    best_ndcg = evaluation.sort_values("ndcg_at_k", ascending=False).iloc[0]
    best_diverse = evaluation.sort_values("diversity", ascending=False).iloc[0]
    print(f"最佳 NDCG: {best_ndcg['model']} (coverage={best_ndcg['coverage']:.3f}, diversity={best_ndcg['diversity']:.3f})")
    print(f"最佳 Diversity: {best_diverse['model']} (coverage={best_diverse['coverage']:.3f}, diversity={best_diverse['diversity']:.3f})")

## 8. 流行度偏差深入分析

In [ ]:
if bias is not None and evaluation is not None:
    merged = bias.merge(evaluation[["model", "ndcg_at_k", "diversity"]], on="model")

    fig, ax = plt.subplots(figsize=(8, 6))
    for _, row in merged.iterrows():
        ax.scatter(row["popular_item_ratio"], row["ndcg_at_k"], s=200, alpha=0.7, label=row["model"])
        ax.annotate(row["model"], (row["popular_item_ratio"], row["ndcg_at_k"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=10)
    ax.set_xlabel("Popular Item Ratio (higher = more bias)")
    ax.set_ylabel("NDCG@K")
    ax.set_title("Popularity Bias vs Recommendation Quality")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("流行度偏差排名 (低到高):")
    for _, row in bias.sort_values("popular_item_ratio").iterrows():
        print(f"  {row['model']:25s} popular_ratio={row['popular_item_ratio']:.3f}  avg_pop={row['avg_rating_count']:.1f}")

## 9. 结论与讨论

### 核心发现

1. **混合模型整体最优**: 结合 CF + 内容 + 流行度的信号，在大部分指标上取得均衡表现。
2. **协同过滤提供最强信号**: 消融实验中 CF-only 的 NDCG 显著高于 content-only 和 popularity-only。
3. **内容特征改善冷启动**: 对于评分稀疏的用户，content-based 模型提供了有意义的推荐。
4. **流行度偏差的双刃剑**: 推荐热门电影提高准确率但降低多样性和新颖性。
5. **用户分群差异明显**: 高活跃用户从 CF 中获益更多，低活跃用户依赖内容和流行度信号。

### 局限性

- 稠密矩阵存储限制了扩展到完整 32M 数据的能力
- 混合权重为固定值，未通过验证集学习优化
- 矩阵分解仅使用基本 SVD，未采用 ALS/SGD 等标准方法
- 冷启动仅依靠内容特征和关键词匹配，精度有限

### 后续改进方向

- 稀疏矩阵 + 增量学习支持更大数据规模
- 超参数自动调优（网格搜索 / 贝叶斯优化）
- 引入 Neural CF 或 LightGCN 等深度学习方法
- 冷启动融入用户画像和多模态信息